# Final eligible geographic extract

This notebook runs extraction and streaming validation from one script. The saved panel has one row per `pixel_id` and monthly date (`year` + `month`), preserves raw curated NDVI/LST values and their missingness, and carries coordinates, fixed-grid labels, eligibility counts, and geographic/calendar split roles. The Parquet is the only generated output.

The source bundle available to this project is the curated annual Parquet export. No separate NumPy arrays are present, so the script compares every selected raw pixel/date/NDVI/LST record against those source Parquet files. Validation processes data in batches and does not load the full panel into memory.

In [1]:
from pathlib import Path
import subprocess
import sys

ROOT = Path.cwd()
if not (ROOT / "scripts").is_dir():
    ROOT = ROOT.parent
assert (ROOT / "scripts/recreate_reference_sample.py").is_file()

subprocess.run(
    [sys.executable, str(ROOT / "scripts/recreate_reference_sample.py")],
    cwd=ROOT,
    check=True,
)


evergreen_locations.parquet schema:
  pixel_id: int64
  lon: double
  lat: double
  green_months_2013_2024: int16
  Rows: 461,695
  Unique pixel IDs: 461,695
  Missing coordinates: 0

evergreen_monthly_2013.parquet schema:
  pixel_id: int64
  year: int32
  month: int32
  ndvi: float
  lst: float
  Rows: 4,155,255; locations: 461,695; duplicate keys: 0

evergreen_monthly_2014.parquet schema:
  pixel_id: int64
  year: int32
  month: int32
  ndvi: float
  lst: float
  Rows: 5,540,340; locations: 461,695; duplicate keys: 0

evergreen_monthly_2015.parquet schema:
  pixel_id: int64
  year: int32
  month: int32
  ndvi: float
  lst: float
  Rows: 5,540,340; locations: 461,695; duplicate keys: 0

evergreen_monthly_2016.parquet schema:
  pixel_id: int64
  year: int32
  month: int32
  ndvi: float
  lst: float
  Rows: 5,540,340; locations: 461,695; duplicate keys: 0

evergreen_monthly_2017.parquet schema:
  pixel_id: int64
  year: int32
  month: int32
  ndvi: float
  lst: float
  Rows: 5,540,340;

CompletedProcess(args=['/opt/anaconda3/bin/python', '/Users/meera/greening_predictive/scripts/recreate_reference_sample.py'], returncode=0)

## Inspect the validated Parquet

The combined script prints validation results and a quality summary. This cell reads only the first five rows, rather than loading the full panel.

In [2]:
import pyarrow.parquet as pq
from IPython.display import display

extract = pq.ParquetFile(ROOT / "outputs/evergreen_reference_sample_recreated.parquet")
print(f"Rows: {extract.metadata.num_rows:,}")
display(extract.schema.to_arrow_schema())
display(next(extract.iter_batches(batch_size=5)).to_pandas())

Rows: 56,354,034


pixel_id: int64
year: int32
month: int32
ndvi: float
lst: float
lon: double
lat: double
easting_m: double
northing_m: double
tile_col: int64
tile_row: int64
tile_id: string
block_col: int64
block_row: int64
block_id: string
geographic_split: string
calendar_role: string
analysis_role: string
green_months_pre2022: int16
valid_ndvi_months_pre2022: int16

,pixel_id,year,month,ndvi,lst,lon,lat,easting_m,northing_m,tile_col,tile_row,tile_id,block_col,block_row,block_id,geographic_split,calendar_role,analysis_role,green_months_pre2022,valid_ndvi_months_pre2022
0,132028,2013,4,0.114884,48.719780,54.419715,24.112893,237726.455919,2.669139e+06,237,2669,237_2669,59,667,59_667,development,fit,model_fit,3,105
1,132028,2013,5,0.104159,47.115021,54.419715,24.112893,237726.455919,2.669139e+06,237,2669,237_2669,59,667,59_667,development,fit,model_fit,3,105
2,132028,2013,6,0.133883,51.815365,54.419715,24.112893,237726.455919,2.669139e+06,237,2669,237_2669,59,667,59_667,development,fit,model_fit,3,105
3,132028,2013,7,0.139458,51.396942,54.419715,24.112893,237726.455919,2.669139e+06,237,2669,237_2669,59,667,59_667,development,fit,model_fit,3,105
4,132028,2013,8,0.140986,52.923943,54.419715,24.112893,237726.455919,2.669139e+06,237,2669,237_2669,59,667,59_667,development,fit,model_fit,3,105


## Saved deliverables

- `outputs/evergreen_reference_sample_recreated.parquet` — validated monthly panel; the only generated output.
- `docs/data_dictionary.csv` — field definitions and missing-value meaning.
- `docs/sample_recreation_report.md` — method, validation, and limitations.